# Tutorial for Stray Light Module

2025-07-08: Algorithms are being actively developed on the stray-light branch by Greg Gilbert

The primary interface for using the stray light module is the class StrayLightAlg  
The primary method of this class is estimate_stray_light()

The algorithm works at the 2D data product level

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kpfpipe.models.level0 import KPF0
from kpfpipe.models.level1 import KPF1
from modules.Utils.kpf_parse import get_datecode
from modules.stray_light.src.alg import StrayLightAlg

%matplotlib inline

To initialize the StrayLightAlg class, we'll need a science frame and a master order mask, both at the 2D level.

We also need to pass the path to a config file. The config file should contain the following fields:  
[PARAM]  
method  
polyorder  
edge_clip  
mask_buffer  

In [ ]:
OBS_ID = 'KP.20240405.49597.71'    # TOI-1184 | V = 11

datecode = get_datecode(OBS_ID)

filepath = os.path.join('/data/2D/', datecode, f'{OBS_ID}_2D.fits')
target_2D = KPF0.from_fits(filepath, data_type='KPF')

filepath = os.path.join('/data/masters/', datecode, f'kpf_{datecode}_order_mask.fits')
master_order_mask = KPF0.from_fits(filepath, data_type='KPF')

stray_light_modeler = StrayLightAlg(target_2D, master_order_mask, '/code/KPF-Pipeline/modules/stray_light/configs/default.cfg')

Here's what our raw science images look like

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,5))

for i, chip in enumerate(['GREEN', 'RED']):
    raw_image = np.array(stray_light_modeler.target_2D[f'{chip}_CCD'])
    thin = 1
    
    im = ax[i].imshow(raw_image,
                      origin='lower',
                      vmin=np.nanpercentile(raw_image, 0.5),
                      vmax=np.nanpercentile(raw_image, 99.5)
                     )
    cbar = fig.colorbar(im, ax=ax[i], shrink=0.85)
    ax[i].set_title(f'{chip}_CCD')
    ax[i].set_xticks([])
    ax[i].set_yticks([])

plt.suptitle(f"{OBS_ID} | TOI-1184 | V=11 | raw image", fontsize=16)
plt.show()

To run the stray light algorithm, simply call the method estimate_stray_light(). By default, the algorithm will use the parameter values defined in the config file, but you can also set these on the fly when developing, as below. 

In either case, you need to specify whether you want to perform stray light estimation on the GREEN or RED ccd. We'll store both of these in a dictionary.

In [ ]:
image = {}
mask = {}
for chip in ['GREEN', 'RED']:
    image[f'{chip}_CCD'], mask[f'{chip}_CCD'] = stray_light_modeler.estimate_stray_light(chip, 
                                                                                         method='polynomial', 
                                                                                         polyorder=5, 
                                                                                         edge_clip=256,
                                                                                         mask_buffer=2
                                                                                        )

The two key components of the algorithm are:
1. Building the inter-order mask, a boolean array which identifies pixels in (1) and out (0) of illuminated orders/orderlets
2. Fitting a smooth model to the inter-order pixels

The method estimate_stray_light() performs both of these tasks and returns both an image of the stray light model and a copy of the boolean inter-order mask used to build fit this model.

The output stray light mask and image are shown below.

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,5))

for i, chip in enumerate(['GREEN', 'RED']):
    im = ax[i].imshow(mask[f'{chip}_CCD'], origin='lower')
    cbar = fig.colorbar(im, ax=ax[i], shrink=0.85)
    ax[i].set_title(f'{chip}_CCD')
    ax[i].set_xticks([])
    ax[i].set_yticks([])

plt.suptitle(f"{OBS_ID} | TOI-1184 | V=11 | inter-order mask", fontsize=16)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,5))

for i, chip in enumerate(['GREEN', 'RED']):
    stray_light = np.array(image[f'{chip}_CCD'])
    thin = 64
    
    im = ax[i].imshow(stray_light[::thin,::thin],
                      origin='lower',
                      vmin=np.nanpercentile(stray_light[::thin,::thin], 0.1),
                      vmax=np.nanpercentile(stray_light[::thin,::thin], 99)
                     )
    cbar = fig.colorbar(im, ax=ax[i], shrink=0.85)
    ax[i].set_title(f'{chip}_CCD')
    ax[i].set_xticks([])
    ax[i].set_yticks([])

plt.suptitle(f"{OBS_ID} | TOI-1184 | V=11 | stray light estimate", fontsize=16)
plt.show()

To assess the quality of the model, we can look at a histogram of residuals in the inter-order region and in the sky fiber, both of which are expected to have a mean of zero.

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,5))
colors = {'GREEN':'C2', 'RED':'C3'}

for i, chip in enumerate(['GREEN', 'RED']):

    data_image = stray_light_modeler.target_2D[f'{chip}_CCD']
    stray_light = np.array(image[f'{chip}_CCD'])

    m = stray_light_modeler.master_order_mask[f'{chip}_CCD'] == 0

    residuals = (data_image-stray_light)[m]
    bins = np.linspace(np.nanpercentile(residuals,0.5),np.nanpercentile(residuals,99.5),50)

    vlines = np.nanpercentile(residuals, [16,50,84])
    ls = [':', '--', ':']

    ax[i].hist(residuals, bins=bins, color=colors[chip])
    for j, line in enumerate(vlines):
        ax[i].axvline(line, color='k', lw=2, ls=ls[j])
    ax[i].set_title(f'{chip} CCD', fontsize=12)
    
plt.suptitle(f"{OBS_ID} | TOI-1184 | V=11 | inter-order residuals", fontsize=16)    
plt.show()

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(16,5))
colors = {'GREEN':'C2', 'RED':'C3'}

for i, chip in enumerate(['GREEN', 'RED']):

    data_image = stray_light_modeler.target_2D[f'{chip}_CCD']
    stray_light = np.array(image[f'{chip}_CCD'])

    m = stray_light_modeler.master_order_mask[f'{chip}_CCD'] == 1

    residuals = (data_image-stray_light)[m]
    bins = np.linspace(np.nanpercentile(residuals,0.5),np.nanpercentile(residuals,99.5),50)

    vlines = np.nanpercentile(residuals, [16,50,84])
    ls = [':', '--', ':']

    ax[i].hist(residuals, bins=bins, color=colors[chip])
    for j, line in enumerate(vlines):
        ax[i].axvline(line, color='k', lw=2, ls=ls[j])    
    ax[i].set_title(f'{chip} CCD', fontsize=12)
    
plt.suptitle("SKY fiber residuals", fontsize=16)    
plt.show()

## Development notes

2025-07-08: Currently supported methods for the stray light model are "zero" (which returns an array of zeros), "mean" (which returns the mean of all inter order pixels, and "polynomial" which fits a 2D polynomial to all intra-order pixels. The order of the polynomial may be set using the keyword argument "polyorder" and the number of CCD edge pixels to exclude can be set by the keyword argment "edge_clip". Future updates will allow for regularization of the polynomial coefficients and incorporation of other smooth fitting bases.

2025-07-09: The inter-order mask is built by converting a master master order mask into a boolean array. Passing the keyword argument "mask_buffer" will expand the illuminated region by N=mask_buffer number of pixels. A good default value mask_buffer=2 will preserve most of the inter-order unilluminated regions but exclude the inter-orderlet regions from the stray light fit. There is also (partially implemented) support for user-selected dark fibers, i.e. orderlets which are not illuminated in the science frame and so should be considered part of the inter-order region.